In [ ]:
# --- Step 1: Install Required Libraries ---
# We need to install PyTorch Geometric and its dependencies to build our Temporal Graph Network.
# This command will install the correct versions for the GPU environment in Colab.
print("Installing specialized libraries for Temporal Graph Networks...")
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
!pip install torch-geometric
!pip install torch-geometric-temporal
print("\nInstallation complete!")

Installing specialized libraries for Temporal Graph Networks...
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu121.html
  Using cached torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl
  Using cached torch_sparse-0.6.18.tar.gz (209 kB)
  Preparing metadata (setup.py) ... done
  Using cached torch_cluster-1.6.3.tar.gz (54 kB)
  Preparing metadata (setup.py) ... done
  Using cached torch_spline_conv-1.2.2.tar.gz (25 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for torch-sparse: filename=torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl size=2906587 sha256=28fae18e3bc3cbcd1091afa5547adcec59369a2cd119bc33f453de3d6a167387
  Stored in directory: /root/.cache/pip/wheels/71/fa/21/bd1d78ce1629aec4ecc924a63b82f6949dda484b6321eac6f2
ERROR: Operation cancelled by user
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)
  Using cached torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl
  Using cache

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- Step 2: Import Libraries and Load Data ---
import pandas as pd
import numpy as np
from datetime import datetime
import torch

print("Loading the prepared temporal data...")

# Define the filename
file_path = 'temporal_events.csv'

# Load the CSV file into a pandas DataFrame
try:
    df = pd.read_csv(file_path, parse_dates=['datetime'])
    print("Successfully loaded 'temporal_events.csv'.")

    # --- Inspect the Data ---
    print("\nHere's a preview of the first 10 events:")
    print(df.head(10))

    print(f"\nTotal number of events in the dataset: {len(df)}")
    print(f"Number of unique users: {df['user'].nunique()}")
    print(f"Number of unique event types: {df['activity'].nunique()}")

except FileNotFoundError:
    print(f"\n--- ERROR ---")
    print(f"File not found: {file_path}")
    print("Please make sure you have uploaded the 'temporal_events.csv' file to the Colab environment.")



Loading the prepared temporal data...
Successfully loaded 'temporal_events.csv'.

Here's a preview of the first 10 events:
             datetime     user    activity
0 2010-01-02 06:49:00  NGF0157       Logon
1 2010-01-02 06:50:00  LRR0148       Logon
2 2010-01-02 06:53:04  LRR0148       Logon
3 2010-01-02 07:00:00  MOH0273       Logon
4 2010-01-02 07:00:00  IRM0931       Logon
5 2010-01-02 07:07:00  LAP0338       Logon
6 2010-01-02 07:08:00  MHH0180       Logon
7 2010-01-02 07:08:00  NOB0181       Logon
8 2010-01-02 07:11:45  LAP0338  sent_email
9 2010-01-02 07:12:16  MOH0273  sent_email

Total number of events in the dataset: 3890218
Number of unique users: 1000
Number of unique event types: 5


In [ ]:
# --- Step 3: Prepare Data for the Temporal Graph Network ---
# The model needs numerical inputs, so we'll map users and activities to integer IDs.

print("Preparing data for the temporal model...")

# Create a mapping from unique user IDs to integers
user_mapping = {user: i for i, user in enumerate(df['user'].unique())}
df['user_id'] = df['user'].map(user_mapping)

# Create a mapping from unique activity types to integers
activity_mapping = {activity: i for i, activity in enumerate(df['activity'].unique())}
df['activity_id'] = df['activity'].map(activity_mapping)

# One-hot encode the activities to create feature vectors for our graph nodes
# Each activity will be a vector of 0s with a single 1.
activity_features = pd.get_dummies(df['activity']).values

# Sort the dataframe by time, which is crucial for a temporal model
df = df.sort_values('datetime').reset_index(drop=True)

print("\nData preprocessing complete!")
print("Created numerical mappings for users and activities.")
print("\nHere's a preview of the processed data with new ID columns:")
print(df.head(10))

# Store the number of unique users and activities for model creation later
num_users = len(user_mapping)
num_activities = len(activity_mapping)
print(f"\nTotal unique users (nodes): {num_users}")
print(f"Total unique activities (features): {num_activities}")



Preparing data for the temporal model...

Data preprocessing complete!
Created numerical mappings for users and activities.

Here's a preview of the processed data with new ID columns:
             datetime     user    activity  user_id  activity_id
0 2010-01-02 06:49:00  NGF0157       Logon        0            0
1 2010-01-02 06:50:00  LRR0148       Logon        1            0
2 2010-01-02 06:53:04  LRR0148       Logon        1            0
3 2010-01-02 07:00:00  MOH0273       Logon        2            0
4 2010-01-02 07:00:00  IRM0931       Logon        3            0
5 2010-01-02 07:07:00  LAP0338       Logon        4            0
6 2010-01-02 07:08:00  MHH0180       Logon        5            0
7 2010-01-02 07:08:00  NOB0181       Logon        6            0
8 2010-01-02 07:11:45  LAP0338  sent_email        4            1
9 2010-01-02 07:12:16  MOH0273  sent_email        2            1

Total unique users (nodes): 1000
Total unique activities (features): 5


In [ ]:
# --- Step 4: Define the Temporal Graph Network (TGN) Model ---
import torch
import torch.nn.functional as F
# --- ROBUST FIX: We will build a custom temporal model using stable core components ---
from torch_geometric.nn import GCNConv
from torch.nn import GRU

class TemporalGCN(torch.nn.Module):
    """
    A custom Temporal Graph Convolutional Network model.
    This model uses a stable GCN layer and a standard GRU layer to process sequences.
    """
    def __init__(self, node_features, num_classes):
        super(TemporalGCN, self).__init__()

        # 1. A Graph Convolutional layer to learn from the graph structure
        self.gcn = GCNConv(node_features, 32)

        # 2. A standard Gated Recurrent Unit (GRU) to learn from the sequence of days
        self.gru = GRU(32, 32) # Input and hidden size are both 32

        # 3. A final linear layer to produce the output
        self.linear = torch.nn.Linear(32, num_classes)

    def forward(self, x, edge_index, h=None):
        # x: node features for the current timestep
        # edge_index: graph connectivity for the current timestep
        # h: hidden state from the previous timestep

        # --- Step A: Process the spatial (graph) information ---
        # Get graph embeddings for the current day's activities
        graph_embedding = self.gcn(x, edge_index)
        graph_embedding = F.relu(graph_embedding)

        # --- Step B: Process the temporal (sequence) information ---
        # The GRU expects input of shape (seq_len, batch_size, input_size)
        # Our "batch" is the set of all users, so we reshape the embedding.
        # (num_users, features) -> (1, num_users, features)
        graph_embedding = graph_embedding.unsqueeze(0)

        # Pass the graph embedding and the previous hidden state to the GRU
        gru_out, h_next = self.gru(graph_embedding, h)

        # Reshape the output back to (num_users, features)
        gru_out = gru_out.squeeze(0)

        # --- Step C: Final Prediction ---
        # Apply the final linear layer
        output = self.linear(gru_out)

        return output, h_next

print("Custom Temporal Graph Network model architecture has been defined successfully.")
print("The model is ready to be trained.")



Custom Temporal Graph Network model architecture has been defined successfully.
The model is ready to be trained.


In [ ]:
# --- Step 5: Create a Generator for Temporal Graph Snapshots ---
from torch_geometric.utils import to_networkx
import networkx as nx

def generate_daily_snapshots(df, num_users, num_features):
    """
    A generator function that yields a graph snapshot for each day in the dataset.
    Each snapshot contains all user activities for that day.
    """
    # --- ROBUST FIX: Handle inconsistent date formats ---
    # The 'format="mixed"' argument tells pandas to intelligently handle different formats.
    # The 'errors="coerce"' will turn any unparseable dates into NaT (Not a Time).
    print("Parsing datetime column with flexible format...")
    df['datetime'] = pd.to_datetime(df['datetime'], format="mixed", errors="coerce")

    # Drop any rows where the date could not be parsed
    original_rows = len(df)
    df.dropna(subset=['datetime'], inplace=True)
    if len(df) < original_rows:
        print(f"Dropped {original_rows - len(df)} rows with invalid date formats.")

    # Group all events by day
    daily_groups = df.groupby(df['datetime'].dt.date)

    print(f"Processing {len(daily_groups)} daily snapshots...")

    for day, group in daily_groups:
        # Node features: A matrix where each row is a user.
        # We start with zeros and will fill in the activities.
        node_features = torch.zeros(num_users, num_features, dtype=torch.float32)

        # For each event on this day, update the user's feature vector
        for _, row in group.iterrows():
            user_idx = row['user_id']
            activity_idx = row['activity_id']
            # Mark the activity that occurred for that user
            node_features[user_idx, activity_idx] = 1

        # Edges: For simplicity, we create a fully connected graph.
        # This means every user is potentially connected to every other user.
        edge_index = torch.combinations(torch.arange(num_users, dtype=torch.long), r=2).t().contiguous()

        # Yield the snapshot for this day
        yield {
            "x": node_features,
            "edge_index": edge_index,
            "day": day
        }

# Create our snapshot generator
daily_snapshot_generator = generate_daily_snapshots(df, num_users, num_activities)

print("\nDaily snapshot generator created.")
print("We are now ready to begin training the model.")




Daily snapshot generator created.
We are now ready to begin training the model.


In [ ]:
# --- Step 6: Train the Temporal Graph Network Model ---
# This is the final and most computationally intensive step.

print("Initializing the model and optimizer...")

# Move model to the GPU if available (which it is in Colab)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create an instance of our model
# The output of the model will be a prediction of the next day's features
model = TemporalGCN(node_features=num_activities, num_classes=num_activities).to(device)

# Use the Adam optimizer, a standard choice for deep learning
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- The Training Loop ---
print("\nStarting model training...")
print("This will take 5-10 minutes. You will see the loss printed for each daily snapshot.")

# We will train for a few "epochs". An epoch is one full pass over the entire dataset.
epochs = 3
for epoch in range(epochs):
    print(f"\n--- Epoch {epoch+1}/{epochs} ---")

    # Reset the snapshot generator for each epoch
    daily_snapshot_generator = generate_daily_snapshots(df, num_users, num_activities)

    # Initialize the hidden state for the recurrent layer at the start of each epoch
    h = None
    total_loss = 0
    num_snapshots = 0

    # Get the first snapshot to start the loop
    try:
        previous_snapshot = next(daily_snapshot_generator)
    except StopIteration:
        print("Dataset is too small to train. Needs at least 2 days.")
        break

    # Loop through the rest of the daily snapshots
    for current_snapshot in daily_snapshot_generator:
        model.train()
        optimizer.zero_grad()

        # Get data for the previous day (input) and current day (target)
        x_prev = previous_snapshot['x'].to(device)
        edge_index_prev = previous_snapshot['edge_index'].to(device)
        y_true = current_snapshot['x'].to(device)

        # Get the model's prediction for the current day's activities
        y_pred, h_next = model(x_prev, edge_index_prev, h)

        # Calculate the loss: how different was the prediction from the actual activities?
        loss = torch.mean((y_pred - y_true)**2)

        # Backpropagate the loss and update the model's weights
        loss.backward()
        optimizer.step()

        # --- ROBUST FIX: Detach the hidden state from the computation graph ---
        # This prevents the error by treating the memory from the last step as a new input.
        h = h_next.detach()

        total_loss += loss.item()
        num_snapshots += 1

        # Print progress every 50 days
        if num_snapshots % 50 == 0:
            print(f"Day {num_snapshots}, Current Loss: {loss.item():.6f}")

        # The current snapshot becomes the previous one for the next iteration
        previous_snapshot = current_snapshot

    avg_loss = total_loss / num_snapshots if num_snapshots > 0 else 0
    print(f"--- End of Epoch {epoch+1} ---")
    print(f"Average training loss for this epoch: {avg_loss:.6f}")

print("\n\nModel training complete!")
print("The model has now learned the patterns of normal daily activity.")



Initializing the model and optimizer...
Using device: cuda

Starting model training...
This will take 5-10 minutes. You will see the loss printed for each daily snapshot.

--- Epoch 1/3 ---
Parsing datetime column with flexible format...
Processing 501 daily snapshots...
Day 50, Current Loss: 0.104992
Day 100, Current Loss: 0.122292
Day 150, Current Loss: 0.328657
Day 200, Current Loss: 0.093158
Day 250, Current Loss: 0.172291
Day 300, Current Loss: 0.115861
Day 350, Current Loss: 0.367161
Day 400, Current Loss: 0.031087
Day 450, Current Loss: 0.119911
Day 500, Current Loss: 0.326077
--- End of Epoch 1 ---
Average training loss for this epoch: 0.151383

--- Epoch 2/3 ---
Parsing datetime column with flexible format...
Processing 501 daily snapshots...
Day 50, Current Loss: 0.033500
Day 100, Current Loss: 0.106920
Day 150, Current Loss: 0.453146
Day 200, Current Loss: 0.082745
Day 250, Current Loss: 0.131327
Day 300, Current Loss: 0.114072
Day 350, Current Loss: 0.354070
Day 400, Curren

In [ ]:
# --- Step 7: Anomaly Detection with the Trained Model ---
# Now that the model has learned what "normal" looks like, we can use it to find anomalies.

print("Starting anomaly detection process...")

# We need the "answer key" to see if we found the real insiders.
# First, let's upload it to our Colab environment.
from google.colab import files
import io

print("\nPlease upload the 'insiders.csv' file.")
uploaded = files.upload()

# Check if the file was uploaded and load it
insider_file_name = next(iter(uploaded))
print(f"\nSuccessfully uploaded {insider_file_name}.")
insiders_df = pd.read_csv(io.BytesIO(uploaded[insider_file_name]))

# Get the list of true insider user IDs from the answer key
true_insiders = set(insiders_df['user'].unique())
print(f"Found {len(true_insiders)} true insiders in the answer key.")


# --- Calculate Anomaly Scores ---
# The anomaly score for each user will be their average reconstruction error over all days.
# A high error means the model was "surprised" by the user's activity.
model.eval() # Put the model in evaluation mode
daily_snapshot_generator = generate_daily_snapshots(df, num_users, num_activities)
h = None
previous_snapshot = next(daily_snapshot_generator)

# A dictionary to store the total error for each user
user_errors = {i: 0.0 for i in range(num_users)}
num_snapshots = 0

for current_snapshot in daily_snapshot_generator:
    x_prev = previous_snapshot['x'].to(device)
    edge_index_prev = previous_snapshot['edge_index'].to(device)
    y_true = current_snapshot['x'].to(device)

    # Get the model's prediction
    y_pred, h_next = model(x_prev, edge_index_prev, h)

    # Calculate the error for each user on this day
    daily_error = torch.mean((y_pred - y_true)**2, dim=1) # Error per user

    # Add this day's error to each user's total
    for i in range(num_users):
        user_errors[i] += daily_error[i].item()

    h = h_next.detach()
    previous_snapshot = current_snapshot
    num_snapshots += 1

# Calculate the average error per user
avg_user_errors = {user: total_error / num_snapshots for user, total_error in user_errors.items()}

# --- Display Results ---
print("\n--- Top 10 Most Anomalous Users (Advanced Model) ---")

# Create a reverse mapping from integer ID back to the original user string ID
reverse_user_mapping = {i: user for user, i in user_mapping.items()}

# Create a results DataFrame
results_list = []
for user_int, avg_error in avg_user_errors.items():
    user_str = reverse_user_mapping[user_int]
    is_insider = user_str in true_insiders
    results_list.append({'user': user_str, 'anomaly_score': avg_error, 'is_insider': is_insider})

results_df = pd.DataFrame(results_list)
top_anomalies = results_df.sort_values('anomaly_score', ascending=False).head(10)

print(top_anomalies)

insiders_caught = top_anomalies['is_insider'].sum()
print(f"\nFound {insiders_caught} true insider(s) in the top 10 anomalies.")

print("\n--- Comparison to Baseline Model ---")
print("The baseline model (Isolation Forest) found 2 true insiders.")
if insiders_caught > 2:
    print(f"SUCCESS: The advanced temporal model found {insiders_caught} insiders, outperforming the baseline!")
elif insiders_caught == 2:
    print("RESULT: The advanced model performed equally to the baseline model.")
else:
    print("ANALYSIS: The advanced model found fewer insiders than the baseline. This could be due to hyperparameter settings or the need for more training epochs.")



Starting anomaly detection process...

Please upload the 'insiders.csv' file.


Saving insiders.csv to insiders.csv

Successfully uploaded insiders.csv.
Found 191 true insiders in the answer key.
Parsing datetime column with flexible format...
Processing 501 daily snapshots...

--- Top 10 Most Anomalous Users (Advanced Model) ---
       user  anomaly_score  is_insider
54  QOS0878       0.372040       False
40  BQS0525       0.364342       False
70  LDD0560       0.350020       False
91  GTD0219       0.346402        True
45  DAR0885       0.344134       False
47  BSS0369       0.343426        True
75  MOS0047       0.341403        True
26  HSB0196       0.332135       False
20  HPH0075       0.329175       False
19  BRS0734       0.328080       False

Found 3 true insider(s) in the top 10 anomalies.

--- Comparison to Baseline Model ---
The baseline model (Isolation Forest) found 2 true insiders.
SUCCESS: The advanced temporal model found 3 insiders, outperforming the baseline!
